In [1]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    KBinsDiscretizer,
    OneHotEncoder,
    PowerTransformer,
    StandardScaler,
)
from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    TunedThresholdClassifierCV,
    cross_validate,
    train_test_split,
)

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)

import warnings; warnings.filterwarnings("ignore")

## Read Data CSV

In [2]:
sessions = pd.read_csv("Raw_Data/challenge2_train.csv")

display(sessions.head())
sessions.info()

print()
display(sessions["Revenue"].value_counts(dropna=False))
print()
display(sessions["Revenue"].value_counts(normalize=True, dropna=False))

,ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,3,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
2,4,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
3,7,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
4,9,0,0.0,0,0.0,2,37.000000,0.00,0.10,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False


<class 'pandas.DataFrame'>
RangeIndex: 9247 entries, 0 to 9246
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       9247 non-null   int64  
 1   Administrative           9247 non-null   int64  
 2   Administrative_Duration  9247 non-null   float64
 3   Informational            9247 non-null   int64  
 4   Informational_Duration   9247 non-null   float64
 5   ProductRelated           9247 non-null   int64  
 6   ProductRelated_Duration  9247 non-null   float64
 7   BounceRates              9247 non-null   float64
 8   ExitRates                9247 non-null   float64
 9   PageValues               9247 non-null   float64
 10  SpecialDay               9247 non-null   float64
 11  Month                    9247 non-null   str    
 12  OperatingSystems         9247 non-null   int64  
 13  Browser                  9247 non-null   int64  
 14  Region                   9247 non-n

Revenue
False    7816
True     1431
Name: count, dtype: int64

Revenue
False    0.845247
True     0.154753
Name: proportion, dtype: float64

## Train/Validation Splits 

In [3]:
X = sessions.drop(columns=["Revenue", "ID"])
y = sessions["Revenue"].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

display(pd.DataFrame({
    "development": y_train.value_counts(normalize=True),
    "holdout": y_valid.value_counts(normalize=True),
}))

,development,holdout
Revenue,,
False,0.845208,0.845405
True,0.154792,0.154595


## Model Pipeline

In [4]:
categorical = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

numerical = [column for column in X_train.columns if column not in categorical]

# 2. Build leakage-free preprocessing per column type
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    # Yeo-Johnson maps skewed continuous features to Gaussian bell curves
    ('power', PowerTransformer(method='yeo-johnson', standardize=True))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Group rare categories (<1% frequency) to prevent zero-variance overfitting
    ('ohe', OneHotEncoder(
        handle_unknown='infrequent_if_exist',
        min_frequency=0.01,
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical),
        ('cat', cat_pipeline, categorical)
    ],
    remainder='drop'
)

### Logistic Regression

In [5]:
logistic_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

fitted_logistic = clone(logistic_pipeline).fit(X_train, y_train)

feature_names = fitted_logistic.named_steps["prep"].get_feature_names_out()
weights = fitted_logistic.named_steps["model"].coef_[0]

logistic_weights = pd.DataFrame({
    "Feature": feature_names,
    "Weight": weights,
})

display(logistic_weights.round(3))

,Feature,Weight
0,num__Administrative,-0.378
1,num__Administrative_Duration,0.074
2,num__Informational,0.195
3,num__Informational_Duration,-0.239
4,num__ProductRelated,-0.119
5,num__ProductRelated_Duration,0.087
6,num__BounceRates,-0.192
7,num__ExitRates,-0.367
8,num__PageValues,1.625
9,num__SpecialDay,-0.036


### Naïve Bayes

In [6]:
# --- 1. Cross-Validation & Scoring Setup ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

# --- 2. Dedicated Naïve Bayes Pipeline & Discretized Numerical Sub-Pipeline ---
# Clone the global preprocessor so overriding 'prep__num' never mutates shared pipelines
Bayes_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("clf", GaussianNB()),
])

discretized_num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("bins", KBinsDiscretizer(encode="onehot-dense", subsample=None)),
])

# --- 3. Multi-Regime Hyperparameter Grid ---
# Regime A: Discretized numerical bins + BernoulliNB (binarize=None)
# Regime B: Default PowerTransformer + BernoulliNB across non-negative z-score cutoffs
# Regime C: Default PowerTransformer + continuous GaussianNB
param_grid = [
    {
        "prep__num": [discretized_num_pipeline],
        "prep__num__bins__n_bins": [3, 5, 8, 10],
        "prep__num__bins__strategy": ["quantile", "uniform"],
        "clf": [BernoulliNB(binarize=None)],
        "clf__alpha": [0.01, 0.1, 0.5, 1.0, 2.0],
        "clf__class_prior": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
    {
        "clf": [BernoulliNB()],
        "clf__alpha": [0.01, 0.1, 0.5, 1.0, 2.0],
        "clf__binarize": [0.0, 0.25, 0.5, 1.0],
        "clf__class_prior": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
    {
        "clf": [GaussianNB()],
        "clf__var_smoothing": np.logspace(-11, -1, 11),
        "clf__priors": [None, [0.8, 0.2], [0.7, 0.3], [0.6, 0.4], [0.5, 0.5]],
    },
]

# --- 4. Fit GridSearchCV Exclusively on Training Data ---
bayes_search = GridSearchCV(
    estimator=Bayes_pipe,
    param_grid=param_grid,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
    return_train_score=True,
    n_jobs=-1,
)
bayes_search.fit(X_train, y_train)

# --- 5. Post-Processing Decision Threshold Optimization ---
# Tune the probability cutoff to maximize Revenue=True F1 via 5-fold stratified CV
tuned_bayes_model = TunedThresholdClassifierCV(
    estimator=bayes_search.best_estimator_,
    scoring=f1_purchase,
    cv=cv,
    refit=True,
)
tuned_bayes_model.fit(X_train, y_train)

# --- 6. Cross-Validation Performance Summary ---
print("Best GridSearch Parameters:", bayes_search.best_params_)
print(f"Pre-Threshold Best CV F1:   {bayes_search.best_score_:.3f}")
print(f"Optimal Decision Threshold: {tuned_bayes_model.best_threshold_:.3f}")
print(f"Post-Threshold Best CV F1:  {tuned_bayes_model.best_score_:.3f}")

# --- 7. Dynamic Learned Parameter Inspection ---
fitted_best_pipe = tuned_bayes_model.estimator_
feature_names = fitted_best_pipe.named_steps["prep"].get_feature_names_out()
best_clf = fitted_best_pipe.named_steps["clf"]

if isinstance(best_clf, GaussianNB):
    learned_params = pd.DataFrame({
        "Feature": feature_names,
        "Mean (y=0)": best_clf.theta_[0],
        "Mean (y=1)": best_clf.theta_[1],
        "Var (y=0)": best_clf.var_[0],
        "Var (y=1)": best_clf.var_[1],
    })
else:
    feature_prob = np.exp(best_clf.feature_log_prob_)
    learned_params = pd.DataFrame({
        "Feature": feature_names,
        "P(Feature=1 | y=0)": feature_prob[0],
        "P(Feature=1 | y=1)": feature_prob[1],
    })

display(learned_params.head(10).round(3))

Best GridSearch Parameters: {'clf': BernoulliNB(), 'clf__alpha': 1.0, 'clf__binarize': 1.0, 'clf__class_prior': [0.8, 0.2]}
Pre-Threshold Best CV F1:   0.651
Optimal Decision Threshold: 0.491
Post-Threshold Best CV F1:  0.650


,Feature,P(Feature=1 | y=0),P(Feature=1 | y=1)
0,num__Administrative,0.235,0.383
1,num__Administrative_Duration,0.213,0.343
2,num__Informational,0.190,0.325
3,num__Informational_Duration,0.171,0.298
4,num__ProductRelated,0.140,0.275
5,num__ProductRelated_Duration,0.129,0.255
6,num__BounceRates,0.214,0.029
7,num__ExitRates,0.224,0.024
8,num__PageValues,0.106,0.820
9,num__SpecialDay,0.117,0.039


In [7]:
# Predictions for training and validation sets using the threshold-tuned model
y_train_pred = tuned_bayes_model.predict(X_train)
y_valid_pred = tuned_bayes_model.predict(X_valid)
y_valid_prob = tuned_bayes_model.predict_proba(X_valid)[:, 1]

# Compare the main classification metrics
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred, zero_division=0),
        recall_score(y_train, y_train_pred, zero_division=0),
        f1_score(y_train, y_train_pred, zero_division=0),
    ],
    "Validation": [
        accuracy_score(y_valid, y_valid_pred),
        precision_score(y_valid, y_valid_pred, zero_division=0),
        recall_score(y_valid, y_valid_pred, zero_division=0),
        f1_score(y_valid, y_valid_pred, zero_division=0),
    ],
})

display(metrics_comparison.round(3))

,Metric,Training,Validation
0,Accuracy,0.866,0.847
1,Precision,0.546,0.503
2,Recall,0.795,0.776
3,F1 Score,0.647,0.611


### K-Nearest Neighbors

In [8]:
knn_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=5)),
])

fitted_knn = clone(knn_pipeline).fit(X_train, y_train)
knn_model = fitted_knn.named_steps["model"]
print(knn_model.weights)

session = X_train.iloc[[0]]
transformed_session = fitted_knn.named_steps["prep"].transform(session)
distances, indices = knn_model.kneighbors(transformed_session)

display(pd.DataFrame({
    "Distance": distances[0],
    "Purchased": y_train.iloc[indices[0]].to_numpy(),
}))

uniform


,Distance,Purchased
0,0.000000,False
1,1.568383,False
2,1.770997,False
3,1.857021,False
4,1.955126,False


## Support Vector Machine

In [9]:

svmlinear_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", SVC(kernel="linear", C=1.0, class_weight="balanced"))
])

svmrbf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced"
    ))
])

fitted_svmlinear = clone(svmlinear_pipe).fit(X_train, y_train)
fitted_svmrbf = clone(svmrbf_pipe).fit(X_train, y_train)

svmlinear_model = fitted_svmlinear.named_steps["model"]
svmrbf_model = fitted_svmrbf.named_steps["model"]

print("Linear SVM support vectors:", svmlinear_model.n_support_)
print("RBF SVM support vectors:", svmrbf_model.n_support_)


Linear SVM support vectors: [1883  345]
RBF SVM support vectors: [1953  548]


## Model Comparison

In [10]:
model_results = []

pipelines = {
    "Logistic Regression": logistic_pipeline,
    "Naïve Bayes": tuned_bayes_model,
    "K-Nearest Neighbors": knn_pipeline,
    "Linear SVM": svmlinear_pipe,
    "RBF SVM": svmrbf_pipe,
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

for model, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=f1_purchase,
        return_train_score=True,
    )

    model_results.append({
        "Model": model,
        "Mean training F1": scores["train_score"].mean(),
        "Mean CV F1": scores["test_score"].mean(),
    })

display(pd.DataFrame(model_results).round(3))

,Model,Mean training F1,Mean CV F1
0,Logistic Regression,0.674,0.662
1,Naïve Bayes,0.647,0.649
2,K-Nearest Neighbors,0.729,0.633
3,Linear SVM,0.681,0.680
4,RBF SVM,0.712,0.672


## Test Predictions

In [13]:
winning_pipeline = knn_pipeline
final_model = clone(winning_pipeline).fit(X, y)

test_sessions = pd.read_csv("Raw_Data/challenge2_test.csv")
X_test = test_sessions[X.columns]

predictions = pd.DataFrame({
    "ID": test_sessions["ID"],
    "Revenue": final_model.predict(X_test),
})

assert len(predictions) == 3083
assert predictions["ID"].is_unique
assert predictions["Revenue"].isin([True, False]).all()

predictions.to_csv("predictions.csv", index=False)
display(predictions.head())

,ID,Revenue
0,2,False
1,5,False
2,6,False
3,8,False
4,10,False
